In [0]:
-- Creates rules that check against a retrieved data frame before data writes to the lakehouse
-- Replace the schema when necessary.
CREATE TABLE IF NOT EXISTS analytics_lakehouse.silver.data_quality_checks (
  table_name STRING,
  rule_name STRING,
  rule_description STRING,
  sql_expression STRING,
  severity STRING,
  check_type STRING,
  is_active BOOLEAN,
  data_source STRING
);

-- check_type guide
--   WHERE  rd_df.where(sql_expression).count()  (counts rows matching the condition (failures))
--   UNIQUE rd_df.groupBy(sql_expression).count().filter(col('count') > 1).count()  (counts duplicate groups)
 
MERGE INTO analytics_lakehouse.silver.data_quality_checks AS target
USING (
  SELECT * FROM (
    VALUES
  --------------------
  -- VEHICLE LISTINGS
  --------------------
  
  -- Primary Key / Identity (CRITICAL)
  ('denver_vehicle_listings', 'pk_not_null', 'Primary key must not be null', 'transaction_listing_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'pk_unique', 'Primary key must be unique', 'transaction_listing_id', 'CRITICAL', 'UNIQUE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'listing_id_not_null', 'Listing ID must not be null', 'listing_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'dealer_id_not_null', 'Dealer ID must not be null', 'dealer_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'vehicle_id_not_null', 'Vehicle ID must not be null', 'vehicle_id IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),

  -- Date Checks (CRITICAL)
  ('denver_vehicle_listings', 'retrieval_date_not_null', 'Retrieval date must not be null', 'retrieval_date IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'retrieval_date_not_future', 'Retrieval date must not be in the future', 'retrieval_date > current_date()', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'first_seen_not_null', 'First seen date must not be null', 'first_seen IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'first_seen_before_retrieval', 'First seen must be on or before retrieval date', 'first_seen > retrieval_date', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),

  -- Vehicle Identification (CRITICAL / WARNING)
  ('denver_vehicle_listings', 'vin_not_null', 'VIN must not be null', 'vin IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'vin_length', 'VIN must be 17 characters', 'vin IS NOT NULL AND LENGTH(vin) != 17', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'year_valid_range', 'Year must be between 1990 and next year', 'year < 1990 OR year > YEAR(current_date()) + 1', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'make_not_null', 'Make must not be null', 'make IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'model_not_null', 'Model must not be null', 'model IS NULL', 'CRITICAL', 'WHERE', true, 'denver_vehicle_listings'),

  -- Pricing (WARNING)
  ('denver_vehicle_listings', 'list_price_positive', 'List price must be positive when present', 'list_price IS NOT NULL AND list_price <= 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'list_price_reasonable', 'List price must be under $500,000', 'list_price IS NOT NULL AND list_price >= 500000', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'current_price_positive', 'Current price must be positive when present', 'current_price IS NOT NULL AND current_price <= 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'one_price_positive', 'One price must be positive when present', 'one_price IS NOT NULL AND one_price <= 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),

  -- Mileage (WARNING)
  ('denver_vehicle_listings', 'mileage_non_negative', 'Mileage must be non-negative when present', 'mileage IS NOT NULL AND mileage < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'mileage_reasonable', 'Mileage must be under 500,000', 'mileage IS NOT NULL AND mileage >= 500000', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),

  -- Derived Metrics (WARNING)
  ('denver_vehicle_listings', 'days_on_market_non_negative', 'Days on market must be non-negative', 'days_on_market IS NOT NULL AND days_on_market < 0', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'days_on_market_reasonable', 'Days on market must be under 730 (2 years)', 'days_on_market IS NOT NULL AND days_on_market >= 730', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),

  -- Dealer Information (WARNING)
  ('denver_vehicle_listings', 'dealer_name_not_null', 'Dealer name must not be null', 'dealer_name IS NULL', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'dealer_state_valid', 'Dealer state must be a 2-letter code', 'dealer_state IS NOT NULL AND LENGTH(dealer_state) != 2', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'dealer_latitude_valid', 'Dealer latitude must be in valid US range', 'dealer_latitude IS NOT NULL AND (dealer_latitude < 24 OR dealer_latitude > 50)', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'dealer_longitude_valid', 'Dealer longitude must be in valid US range', 'dealer_longitude IS NOT NULL AND (dealer_longitude < -125 OR dealer_longitude > -66)', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'dealer_rating_range', 'Dealer average rating must be between 0 and 5', 'dealer_average_rating < 0 OR dealer_average_rating > 5', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),

  -- Enum/Category (WARNING)
  ('denver_vehicle_listings', 'value_variance_groups_valid', 'Value variance group must be a known category', 'value_variance_groups IS NOT NULL AND value_variance_groups NOT IN (''Great'', ''Good'', ''Fair'', ''Bad'')', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'body_type_not_null', 'Body type must not be null', 'body_type IS NULL', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  ('denver_vehicle_listings', 'fuel_not_null', 'Fuel type must not be null', 'fuel IS NULL', 'WARNING', 'WHERE', true, 'denver_vehicle_listings'),
  
  --------------------
  -- QUICKBOOKS
  --------------------

  -- qb_account
  ('qb_account', 'null_account_id', 'Primary key must not be null', 'account_id IS NULL', 'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_account', 'duplicate_account_id', 'Primary key must be unique', 'account_id', 'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_account', 'null_account_name', 'Account name should not be null', 'account_name IS NULL', 'WARNING',  'WHERE',  TRUE, 'QB'),
  ('qb_account', 'null_account_type', 'Account type should not be null', 'account_type IS NULL', 'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_class
  ('qb_class', 'null_class_id',            'Primary key must not be null',                     'class_id IS NULL',             'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_class', 'duplicate_class_id',       'Primary key must be unique',                       'class_id',                     'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_class', 'null_class_name',          'Class name should not be null',                    'class_name IS NULL',           'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_customer
  ('qb_customer', 'null_customer_id',      'Primary key must not be null',                     'customer_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_customer', 'duplicate_customer_id', 'Primary key must be unique',                       'customer_id',                  'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_customer', 'null_customer_name',    'Customer name should not be null',                 'customer_name IS NULL',        'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_customer_type
  ('qb_customer_type', 'null_customer_type_id',      'Primary key must not be null',            'customer_type_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_customer_type', 'duplicate_customer_type_id', 'Primary key must be unique',              'customer_type_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_employee
  ('qb_employee', 'null_employee_id',      'Primary key must not be null',                     'employee_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_employee', 'duplicate_employee_id', 'Primary key must be unique',                       'employee_id',                  'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_employee', 'null_employee_name',    'Employee full name should not be null',             'employee_full_name IS NULL',   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_discount
  ('qb_item_discount', 'null_item_discount_id',      'Primary key must not be null',            'item_discount_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_discount', 'duplicate_item_discount_id', 'Primary key must be unique',              'item_discount_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_discount', 'null_item_discount_name',    'Item discount name should not be null',   'item_discount_name IS NULL',   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_fixed_asset
  ('qb_item_fixed_asset', 'null_item_fixed_asset_id',      'Primary key must not be null',      'item_fixed_asset_id IS NULL',  'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_fixed_asset', 'duplicate_item_fixed_asset_id', 'Primary key must be unique',        'item_fixed_asset_id',          'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_fixed_asset', 'null_item_fixed_asset_name',    'Item name should not be null',      'item_fixed_asset_name IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_fixed_asset', 'negative_cost_basis',           'Cost basis should not be negative', 'cost_basis < 0',               'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_group
  ('qb_item_group', 'null_item_group_id',      'Primary key must not be null',                  'item_group_id IS NULL',        'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_group', 'duplicate_item_group_id', 'Primary key must be unique',                    'item_group_id',                'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_group', 'null_item_group_name',    'Item group name should not be null',            'item_group_name IS NULL',      'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_inventory
  ('qb_item_inventory', 'null_item_inventory_id',      'Primary key must not be null',          'item_inventory_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_inventory', 'duplicate_item_inventory_id', 'Primary key must be unique',            'item_inventory_id',            'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_inventory', 'null_item_inventory_name',    'Item name should not be null',          'item_inventory_name IS NULL',  'WARNING',  'WHERE',  TRUE, 'QB'),
  ('qb_item_inventory', 'negative_quantity_on_hand',   'Quantity on hand should not be negative','quantity_on_hand < 0',         'WARNING',  'WHERE',  TRUE, 'QB'),
  ('qb_item_inventory', 'negative_average_cost',       'Average cost should not be negative',   'average_cost < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_non_inventory
  ('qb_item_non_inventory', 'null_item_non_inventory_id',      'Primary key must not be null',  'item_non_inventory_id IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_non_inventory', 'duplicate_item_non_inventory_id', 'Primary key must be unique',    'item_non_inventory_id',        'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_non_inventory', 'null_item_non_inventory_name',    'Item name should not be null',  'item_non_inventory_name IS NULL','WARNING','WHERE',  TRUE, 'QB'),
 
  -- qb_item_other_charge
  ('qb_item_other_charge', 'null_item_other_charge_id',      'Primary key must not be null',    'item_other_charge_id IS NULL', 'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_other_charge', 'duplicate_item_other_charge_id', 'Primary key must be unique',      'item_other_charge_id',         'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_other_charge', 'null_item_other_charge_name',    'Item name should not be null',    'item_other_charge_name IS NULL','WARNING', 'WHERE',  TRUE, 'QB'),
 
  -- qb_item_payment
  ('qb_item_payment', 'null_item_payment_id',      'Primary key must not be null',              'item_payment_id IS NULL',      'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_payment', 'duplicate_item_payment_id', 'Primary key must be unique',                'item_payment_id',              'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_payment', 'null_item_payment_name',    'Item name should not be null',              'item_payment_name IS NULL',    'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_sales_tax
  ('qb_item_sales_tax', 'null_item_sales_tax_id',      'Primary key must not be null',          'item_sales_tax_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_sales_tax', 'duplicate_item_sales_tax_id', 'Primary key must be unique',            'item_sales_tax_id',            'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_sales_tax', 'null_item_sales_tax_name',    'Item name should not be null',          'item_sales_tax_name IS NULL',  'WARNING',  'WHERE',  TRUE, 'QB'),
  ('qb_item_sales_tax', 'negative_sales_tax_rate',     'Sales tax rate should not be negative', 'sales_tax_rate < 0',           'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_service
  ('qb_item_service', 'null_item_service_id',      'Primary key must not be null',              'item_service_id IS NULL',      'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_service', 'duplicate_item_service_id', 'Primary key must be unique',                'item_service_id',              'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_service', 'null_item_service_name',    'Item name should not be null',              'item_service_name IS NULL',    'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_subtotal
  ('qb_item_subtotal', 'null_item_subtotal_id',      'Primary key must not be null',            'item_subtotal_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_subtotal', 'duplicate_item_subtotal_id', 'Primary key must be unique',              'item_subtotal_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_subtotal', 'null_item_subtotal_name',    'Item name should not be null',            'item_subtotal_name IS NULL',   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_job_type
  ('qb_job_type', 'null_job_type_id',      'Primary key must not be null',                     'job_type_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_job_type', 'duplicate_job_type_id', 'Primary key must be unique',                       'job_type_id',                  'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_other_name
  ('qb_other_name', 'null_other_name_id',      'Primary key must not be null',                 'other_name_id IS NULL',        'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_other_name', 'duplicate_other_name_id', 'Primary key must be unique',                   'other_name_id',                'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_payment_method
  ('qb_payment_method', 'null_payment_method_id',      'Primary key must not be null',          'payment_method_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_payment_method', 'duplicate_payment_method_id', 'Primary key must be unique',            'payment_method_id',            'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_payroll_item_wage
  ('qb_payroll_item_wage', 'null_payroll_item_wage_id',      'Primary key must not be null',    'payroll_item_wage_id IS NULL', 'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_payroll_item_wage', 'duplicate_payroll_item_wage_id', 'Primary key must be unique',      'payroll_item_wage_id',         'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_payroll_item_nonwage
  ('qb_payroll_item_nonwage', 'null_payroll_item_nonwage_id',      'Primary key must not be null','payroll_item_nonwage_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_payroll_item_nonwage', 'duplicate_payroll_item_nonwage_id', 'Primary key must be unique',  'payroll_item_nonwage_id',       'CRITICAL','UNIQUE',TRUE, 'QB'),
 
  -- qb_price_level
  ('qb_price_level', 'null_price_level_id',      'Primary key must not be null',                'price_level_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_price_level', 'duplicate_price_level_id', 'Primary key must be unique',                  'price_level_id',               'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_sales_tax_code
  ('qb_sales_tax_code', 'null_sales_tax_code_id',      'Primary key must not be null',          'sales_tax_code_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_tax_code', 'duplicate_sales_tax_code_id', 'Primary key must be unique',            'sales_tax_code_id',            'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_ship_method
  ('qb_ship_method', 'null_ship_method_id',      'Primary key must not be null',                'ship_method_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_ship_method', 'duplicate_ship_method_id', 'Primary key must be unique',                  'ship_method_id',               'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_unit_of_measure_set
  ('qb_unit_of_measure_set', 'null_uom_id',      'Primary key must not be null',                'unit_of_measure_set_id IS NULL','CRITICAL','WHERE',  TRUE, 'QB'),
  ('qb_unit_of_measure_set', 'duplicate_uom_id', 'Primary key must be unique',                  'unit_of_measure_set_id',        'CRITICAL','UNIQUE', TRUE, 'QB'),
 
  -- qb_vendor
  ('qb_vendor', 'null_vendor_id',      'Primary key must not be null',                         'vendor_id IS NULL',            'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor', 'duplicate_vendor_id', 'Primary key must be unique',                           'vendor_id',                    'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_vendor', 'null_vendor_name',    'Vendor name should not be null',                       'vendor_name IS NULL',          'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_vendor_type
  ('qb_vendor_type', 'null_vendor_type_id',      'Primary key must not be null',                'vendor_type_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_type', 'duplicate_vendor_type_id', 'Primary key must be unique',                  'vendor_type_id',               'CRITICAL', 'UNIQUE', TRUE, 'QB'),
 
  -- qb_bill
  ('qb_bill', 'null_bill_id',          'Primary key must not be null',                         'bill_id IS NULL',              'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill', 'duplicate_bill_id',     'Primary key must be unique',                           'bill_id',                      'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_bill', 'null_transaction_date', 'Transaction date must not be null',                    'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill', 'null_vendor_id',        'Vendor FK must not be null',                           'vendor_id IS NULL',            'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill', 'negative_amount_due',   'Amount due should not be negative',                    'amount_due < 0',              'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_bill_payment_check
  ('qb_bill_payment_check', 'null_bpc_id',            'Primary key must not be null',           'bill_payment_check_id IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_payment_check', 'duplicate_bpc_id',       'Primary key must be unique',             'bill_payment_check_id',        'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_bill_payment_check', 'null_transaction_date',  'Transaction date must not be null',      'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_payment_check', 'negative_amount',        'Amount should not be negative',          'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_check
  ('qb_check', 'null_check_id',          'Primary key must not be null',                       'check_id IS NULL',             'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_check', 'duplicate_check_id',     'Primary key must be unique',                         'check_id',                     'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_check', 'null_transaction_date',  'Transaction date must not be null',                  'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_check', 'negative_amount',        'Amount should not be negative',                      'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_credit_card_charge
  ('qb_credit_card_charge', 'null_ccc_id',            'Primary key must not be null',           'credit_card_charge_id IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_card_charge', 'duplicate_ccc_id',       'Primary key must be unique',             'credit_card_charge_id',        'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_credit_card_charge', 'null_transaction_date',  'Transaction date must not be null',      'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_card_charge', 'negative_amount',        'Amount should not be negative',          'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_credit_memo
  ('qb_credit_memo', 'null_cm_id',              'Primary key must not be null',                 'credit_memo_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_memo', 'duplicate_cm_id',         'Primary key must be unique',                   'credit_memo_id',               'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_credit_memo', 'null_transaction_date',   'Transaction date must not be null',            'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_memo', 'negative_total_amount',   'Total amount should not be negative',          'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_deposit
  ('qb_deposit', 'null_deposit_id',        'Primary key must not be null',                     'deposit_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_deposit', 'duplicate_deposit_id',   'Primary key must be unique',                       'deposit_id',                   'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_deposit', 'null_transaction_date',  'Transaction date must not be null',                'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_deposit', 'negative_deposit_amount','Deposit amount should not be negative',            'deposit_amount < 0',           'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_estimate
  ('qb_estimate', 'null_estimate_id',        'Primary key must not be null',                   'estimate_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_estimate', 'duplicate_estimate_id',   'Primary key must be unique',                     'estimate_id',                  'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_estimate', 'null_transaction_date',   'Transaction date must not be null',              'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_estimate', 'negative_total_amount',   'Total amount should not be negative',            'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_inventory_adjustment
  ('qb_inventory_adjustment', 'null_ia_id',             'Primary key must not be null',         'inventory_adjustment_id IS NULL','CRITICAL','WHERE', TRUE, 'QB'),
  ('qb_inventory_adjustment', 'duplicate_ia_id',        'Primary key must be unique',           'inventory_adjustment_id',       'CRITICAL','UNIQUE', TRUE, 'QB'),
  ('qb_inventory_adjustment', 'null_transaction_date',  'Transaction date must not be null',    'transaction_date IS NULL',      'CRITICAL','WHERE',  TRUE, 'QB'),
 
  -- qb_invoice
  ('qb_invoice', 'null_invoice_id',        'Primary key must not be null',                     'invoice_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_invoice', 'duplicate_invoice_id',   'Primary key must be unique',                       'invoice_id',                   'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_invoice', 'null_transaction_date',  'Transaction date must not be null',                'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_invoice', 'null_customer_id',       'Customer FK must not be null',                     'customer_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_invoice', 'negative_subtotal',      'Subtotal should not be negative',                  'subtotal_amount < 0',          'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_receipt
  ('qb_item_receipt', 'null_ir_id',              'Primary key must not be null',                'item_receipt_id IS NULL',      'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_receipt', 'duplicate_ir_id',         'Primary key must be unique',                  'item_receipt_id',              'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_item_receipt', 'null_transaction_date',   'Transaction date must not be null',           'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
 
  -- qb_journal_entry
  ('qb_journal_entry', 'null_je_id',              'Primary key must not be null',               'journal_entry_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_journal_entry', 'duplicate_je_id',         'Primary key must be unique',                 'journal_entry_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_journal_entry', 'null_transaction_date',   'Transaction date must not be null',          'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
 
  -- qb_purchase_order
  ('qb_purchase_order', 'null_po_id',              'Primary key must not be null',              'purchase_order_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_purchase_order', 'duplicate_po_id',         'Primary key must be unique',                'purchase_order_id',            'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_purchase_order', 'null_transaction_date',   'Transaction date must not be null',         'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_purchase_order', 'null_vendor_id',          'Vendor FK must not be null',                'vendor_id IS NULL',            'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_purchase_order', 'negative_total_amount',   'Total amount should not be negative',       'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_receive_payment
  ('qb_receive_payment', 'null_rp_id',              'Primary key must not be null',             'receive_payment_id IS NULL',   'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_receive_payment', 'duplicate_rp_id',         'Primary key must be unique',               'receive_payment_id',           'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_receive_payment', 'null_transaction_date',   'Transaction date must not be null',        'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_receive_payment', 'null_customer_id',        'Customer FK must not be null',             'customer_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_receive_payment', 'negative_total_amount',   'Total amount should not be negative',      'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_order
  ('qb_sales_order', 'null_so_id',              'Primary key must not be null',                 'sales_order_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_order', 'duplicate_so_id',         'Primary key must be unique',                   'sales_order_id',               'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_sales_order', 'null_transaction_date',   'Transaction date must not be null',            'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_order', 'negative_total_amount',   'Total amount should not be negative',          'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_receipt
  ('qb_sales_receipt', 'null_sr_id',              'Primary key must not be null',               'sales_receipt_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_receipt', 'duplicate_sr_id',         'Primary key must be unique',                 'sales_receipt_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_sales_receipt', 'null_transaction_date',   'Transaction date must not be null',          'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_receipt', 'negative_total_amount',   'Total amount should not be negative',        'total_amount < 0',             'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_tax_payment_check
  ('qb_sales_tax_payment_check', 'null_stpc_id',            'Primary key must not be null',     'sales_tax_payment_check_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_sales_tax_payment_check', 'duplicate_stpc_id',       'Primary key must be unique',       'sales_tax_payment_check_id',        'CRITICAL','UNIQUE',TRUE, 'QB'),
  ('qb_sales_tax_payment_check', 'null_transaction_date',   'Transaction date must not be null','transaction_date IS NULL',          'CRITICAL','WHERE', TRUE, 'QB'),
 
  -- qb_time_tracking
  ('qb_time_tracking', 'null_tt_id',              'Primary key must not be null',               'time_tracking_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_time_tracking', 'duplicate_tt_id',         'Primary key must be unique',                 'time_tracking_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_time_tracking', 'null_transaction_date',   'Transaction date must not be null',          'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_time_tracking', 'negative_duration',       'Duration should not be negative',            'duration_hours < 0',           'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_vendor_credit
  ('qb_vendor_credit', 'null_vc_id',              'Primary key must not be null',               'vendor_credit_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_credit', 'duplicate_vc_id',         'Primary key must be unique',                 'vendor_credit_id',             'CRITICAL', 'UNIQUE', TRUE, 'QB'),
  ('qb_vendor_credit', 'null_transaction_date',   'Transaction date must not be null',          'transaction_date IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_credit', 'negative_credit_amount',  'Credit amount should not be negative',       'credit_amount < 0',            'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_bill_lines
  ('qb_bill_lines', 'null_parent_id',       'Parent FK must not be null',                      'bill_id IS NULL',              'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_lines', 'null_account_id',      'Account FK must not be null',                     'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_lines', 'negative_amount',      'Line amount should not be negative',              'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_bill_payment_check_lines
  ('qb_bill_payment_check_lines', 'null_parent_id',   'Parent FK must not be null',            'bill_payment_check_id IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_payment_check_lines', 'negative_amount',  'Line amount should not be negative',    'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_check_lines
  ('qb_check_lines', 'null_parent_id',      'Parent FK must not be null',                      'check_id IS NULL',             'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_check_lines', 'null_account_id',     'Account FK must not be null',                     'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_check_lines', 'negative_amount',     'Line amount should not be negative',              'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_credit_card_charge_lines
  ('qb_credit_card_charge_lines', 'null_parent_id',   'Parent FK must not be null',            'credit_card_charge_id IS NULL','CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_card_charge_lines', 'null_account_id',  'Account FK must not be null',           'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_card_charge_lines', 'negative_amount',  'Line amount should not be negative',    'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_credit_memo_lines
  ('qb_credit_memo_lines', 'null_parent_id',    'Parent FK must not be null',                  'credit_memo_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_memo_lines', 'negative_amount',   'Line amount should not be negative',          'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_deposit_lines
  ('qb_deposit_lines', 'null_parent_id',     'Parent FK must not be null',                     'deposit_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_deposit_lines', 'null_account_id',    'Account FK must not be null',                    'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_deposit_lines', 'negative_amount',    'Line amount should not be negative',             'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_estimate_lines
  ('qb_estimate_lines', 'null_parent_id',    'Parent FK must not be null',                     'estimate_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_estimate_lines', 'negative_amount',   'Line amount should not be negative',             'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_inventory_adjustment_lines
  ('qb_inventory_adjustment_lines', 'null_parent_id',   'Parent FK must not be null',          'inventory_adjustment_id IS NULL','CRITICAL','WHERE',  TRUE, 'QB'),
 
  -- qb_invoice_lines
  ('qb_invoice_lines', 'null_parent_id',     'Parent FK must not be null',                     'invoice_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_invoice_lines', 'negative_amount',    'Line amount should not be negative',             'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_item_receipt_lines
  ('qb_item_receipt_lines', 'null_parent_id',  'Parent FK must not be null',                   'item_receipt_id IS NULL',      'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_receipt_lines', 'negative_amount', 'Line amount should not be negative',           'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_journal_entry_lines
  ('qb_journal_entry_lines', 'null_parent_id',  'Parent FK must not be null',                  'journal_entry_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_journal_entry_lines', 'null_account_id', 'Account FK must not be null',                 'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
 
  -- qb_purchase_order_lines
  ('qb_purchase_order_lines', 'null_parent_id',  'Parent FK must not be null',                 'purchase_order_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_purchase_order_lines', 'negative_amount', 'Line amount should not be negative',         'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_receive_payment_lines
  ('qb_receive_payment_lines', 'null_parent_id',  'Parent FK must not be null',                'receive_payment_id IS NULL',   'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_receive_payment_lines', 'negative_amount', 'Line amount should not be negative',        'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_order_lines
  ('qb_sales_order_lines', 'null_parent_id',   'Parent FK must not be null',                   'sales_order_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_order_lines', 'negative_amount',  'Line amount should not be negative',           'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_receipt_lines
  ('qb_sales_receipt_lines', 'null_parent_id',   'Parent FK must not be null',                 'sales_receipt_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_receipt_lines', 'negative_amount',  'Line amount should not be negative',         'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- qb_sales_tax_payment_check_lines
  ('qb_sales_tax_payment_check_lines', 'null_parent_id',  'Parent FK must not be null',        'sales_tax_payment_check_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
 
  -- qb_vendor_credit_lines
  ('qb_vendor_credit_lines', 'null_parent_id',   'Parent FK must not be null',                 'vendor_credit_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_credit_lines', 'null_account_id',  'Account FK must not be null',                'account_id IS NULL',           'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_credit_lines', 'negative_amount',  'Line amount should not be negative',         'amount < 0',                   'WARNING',  'WHERE',  TRUE, 'QB'),
 
  -- ============================================================
  -- LINKED TRANSACTION TABLES
  -- ============================================================
 
  -- qb_bill_linked_transactions
  ('qb_bill_linked_transactions', 'null_parent_id',      'Parent FK must not be null',          'bill_id IS NULL',              'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_bill_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE', TRUE, 'QB'),
  ('qb_bill_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_check_linked_transactions
  ('qb_check_linked_transactions', 'null_parent_id',      'Parent FK must not be null',         'check_id IS NULL',             'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_check_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE', TRUE, 'QB'),
  ('qb_check_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_credit_memo_linked_transactions
  ('qb_credit_memo_linked_transactions', 'null_parent_id',      'Parent FK must not be null',   'credit_memo_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_credit_memo_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_credit_memo_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_estimate_linked_transactions
  ('qb_estimate_linked_transactions', 'null_parent_id',      'Parent FK must not be null',      'estimate_id IS NULL',          'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_estimate_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_estimate_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_item_receipt_linked_transactions
  ('qb_item_receipt_linked_transactions', 'null_parent_id',      'Parent FK must not be null',  'item_receipt_id IS NULL',      'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_item_receipt_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_item_receipt_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_purchase_order_linked_transactions
  ('qb_purchase_order_linked_transactions', 'null_parent_id',      'Parent FK must not be null','purchase_order_id IS NULL',    'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_purchase_order_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_purchase_order_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_sales_order_linked_transactions
  ('qb_sales_order_linked_transactions', 'null_parent_id',      'Parent FK must not be null',   'sales_order_id IS NULL',       'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_sales_order_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_sales_order_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB'),
 
  -- qb_vendor_credit_linked_transactions
  ('qb_vendor_credit_linked_transactions', 'null_parent_id',      'Parent FK must not be null', 'vendor_credit_id IS NULL',     'CRITICAL', 'WHERE',  TRUE, 'QB'),
  ('qb_vendor_credit_linked_transactions', 'null_linked_txn_id',  'Linked transaction ID must not be null','linked_transaction_id IS NULL','CRITICAL','WHERE',TRUE, 'QB'),
  ('qb_vendor_credit_linked_transactions', 'null_linked_txn_type','Linked transaction type should not be null','linked_transaction_type IS NULL','WARNING','WHERE',TRUE, 'QB')
  ) AS v (table_name, rule_name, rule_description, sql_expression, severity, check_type, is_active, data_source)
) AS source
ON target.table_name = source.table_name
   AND target.rule_name = source.rule_name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *